In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.sparse import load_npz, csr_matrix, save_npz
from sklearn.metrics.pairwise import cosine_similarity

import joblib
import os

print("Bibliothèques chargées avec succès.")

Bibliothèques chargées avec succès.


In [2]:
# Matrice TF-IDF des articles
tfidf_matrix = load_npz(
    "../data/processed/tfidf_matrix.npz"
)

# Catalogue des articles
news_catalog = pd.read_csv(
    "../data/processed/news_catalog.csv"
)

# Historiques utilisateurs
train_history = pd.read_pickle(
    "../data/processed/train_history.pkl"
)

dev_history = pd.read_pickle(
    "../data/processed/dev_history.pkl"
)

# Vectoriseur TF-IDF
tfidf_vectorizer = joblib.load(
    "../data/processed/tfidf_vectorizer.pkl"
)

# Mapping article -> indice
import json

with open("../data/processed/mappings.json", "r") as f:
    mappings = json.load(f)

item_to_idx = mappings["item_to_idx"]

print("Données chargées avec succès.")

Données chargées avec succès.


In [3]:
print("===== VÉRIFICATION DES DONNÉES =====")

print("Catalogue :", news_catalog.shape)
print("TF-IDF :", tfidf_matrix.shape)

print("Historiques train :", train_history.shape)
print("Historiques dev :", dev_history.shape)

print("Nombre d'articles dans le mapping :", len(item_to_idx))
print("Nombre de termes :", len(tfidf_vectorizer.get_feature_names_out()))

===== VÉRIFICATION DES DONNÉES =====
Catalogue : (65238, 9)
TF-IDF : (65238, 20000)
Historiques train : (156965, 6)
Historiques dev : (73152, 6)
Nombre d'articles dans le mapping : 65238
Nombre de termes : 20000


In [4]:
# Vérifier que chaque article du catalogue possède le bon indice
catalog_ids = news_catalog["news_id"].astype(str).tolist()

indices = [
    int(item_to_idx[news_id])
    for news_id in catalog_ids
]

expected_indices = list(range(len(catalog_ids)))

assert indices == expected_indices, (
    "Erreur : le mapping ne correspond pas à l'ordre du catalogue."
)

assert tfidf_matrix.shape[0] == len(catalog_ids)

print("Alignement catalogue / mapping / TF-IDF vérifié !")

Alignement catalogue / mapping / TF-IDF vérifié !


In [5]:
def build_user_profile(history_indices, tfidf_matrix):
    """
    Construit le profil d'un utilisateur en faisant
    la moyenne des vecteurs TF-IDF de son historique.

    Parameters
    ----------
    history_indices : list
        Indices des articles consultés par l'utilisateur.

    tfidf_matrix : csr_matrix
        Matrice TF-IDF de tous les articles.

    Returns
    -------
    csr_matrix
        Vecteur de profil utilisateur de dimension (1, nombre de termes).
    """

    # Garder uniquement les indices valides
    valid_indices = [
        idx for idx in history_indices
        if 0 <= idx < tfidf_matrix.shape[0]
    ]

    # Si l'historique est vide, retourner un profil nul
    if len(valid_indices) == 0:
        return csr_matrix((1, tfidf_matrix.shape[1]))

    # Récupérer les vecteurs TF-IDF des articles consultés
    article_vectors = tfidf_matrix[valid_indices]

    # Calculer leur moyenne en conservant une matrice creuse
    user_profile = csr_matrix(
        article_vectors.sum(axis=0)
    ) / len(valid_indices)

    return user_profile

In [6]:
# Choisir le premier historique non vide
example_row = train_history[
    train_history["history_length"] > 0
].iloc[0]

example_user = example_row["user_id"]
example_history = example_row["history_indices"]

# Construire le profil
example_profile = build_user_profile(
    example_history,
    tfidf_matrix
)

print("===== PROFIL UTILISATEUR =====")

print("Utilisateur :", example_user)
print("Nombre d'articles consultés :", len(example_history))
print("Dimensions du profil :", example_profile.shape)
print("Type :", type(example_profile))
print("Nombre de termes non nuls :", example_profile.nnz)

===== PROFIL UTILISATEUR =====
Utilisateur : U13740
Nombre d'articles consultés : 9
Dimensions du profil : (1, 20000)
Type : <class 'scipy.sparse._csr.csr_matrix'>
Nombre de termes non nuls : 204


In [ ]:
# Récupérer les noms des termes TF-IDF
feature_names = tfidf_vectorizer.get_feature_names_out()

# Récupérer les valeurs du profil
profile_values = example_profile.toarray().flatten()

# Identifier les 15 termes avec les poids les plus élevés
top_indices = np.argsort(profile_values)[-15:][::-1]

top_terms = pd.DataFrame({
    "term": feature_names[top_indices],
    "tfidf_weight": profile_values[top_indices]
})

print("===== CENTRES D'INTÉRÊT DE L'UTILISATEUR =====")
display(top_terms)



===== CENTRES D'INTÉRÊT DE L'UTILISATEUR =====


,term,tfidf_weight
0,donnell,0.056162
1,barbara,0.055414
2,kay hagan,0.046943
3,hagan,0.046189
4,evans,0.046148
5,speaking,0.044820
6,wedding,0.043558
7,joe biden,0.042702
8,yankees,0.041534
9,attendants,0.041267


In [8]:
os.makedirs("../data/processed", exist_ok=True)

save_npz(
    "../data/processed/example_user_profile.npz",
    example_profile
)

print("Profil utilisateur exemple sauvegardé.")

Profil utilisateur exemple sauvegardé.


In [9]:
# Calculer la similarité entre le profil utilisateur
# et tous les articles du catalogue

similarity_scores = cosine_similarity(
    example_profile,
    tfidf_matrix
).flatten()

print("===== SCORES DE SIMILARITÉ =====")

print("Nombre de scores :", len(similarity_scores))
print("Score minimum :", similarity_scores.min())
print("Score maximum :", similarity_scores.max())
print("Score moyen :", similarity_scores.mean())

===== SCORES DE SIMILARITÉ =====
Nombre de scores : 65238
Score minimum : 0.0
Score maximum : 0.35046081934641854
Score moyen : 0.016483360762118756


In [10]:
# Créer un tableau contenant les articles et leurs scores
recommendation_scores = news_catalog[
    ["news_id", "category", "subcategory", "title"]
].copy()

# Ajouter le score de similarité
recommendation_scores["similarity_score"] = similarity_scores

# Trier par score décroissant
recommendation_scores = recommendation_scores.sort_values(
    by="similarity_score",
    ascending=False
).reset_index(drop=True)

print("===== TOP 10 ARTICLES LES PLUS SIMILAIRES =====")

display(recommendation_scores.head(10))

===== TOP 10 ARTICLES LES PLUS SIMILAIRES =====


,news_id,category,subcategory,title,similarity_score
0,N19347,news,newspolitics,Former US Senator Kay Hagan dead at 66,0.350461
1,N55189,tv,tvnews,"'Wheel Of Fortune' Guest Delivers Hilarious, O...",0.348271
2,N31801,news,newspolitics,Joe Biden reportedly denied Communion at a Sou...,0.346699
3,N34694,tv,tvnews,Rosie O'Donnell: Barbara Walters Isn't 'Up to ...,0.341490
4,N10414,movies,movienews,"Robert Evans, 'Chinatown' Producer and Paramou...",0.335511
5,N18445,sports,football_ncaa,Michigan sends breakup tweet to Notre Dame as ...,0.333920
6,N45794,news,newscrime,Four flight attendants were arrested in Miami'...,0.333379
7,N42782,sports,baseball_mlb,Three takeaways from Yankees' ALCS Game 5 vict...,0.332728
8,N63302,lifestyle,lifestylebuzz,This Wedding Photo of a Canine Best Man Captur...,0.332204
9,N61651,news,newspolitics,Former U.S. Sen. Kay Hagan dead at 66,0.272054


In [11]:
# Récupérer les indices des articles déjà consultés
read_indices = set(example_history)

# Créer une liste des indices de tous les articles
all_indices = np.arange(len(news_catalog))

# Garder uniquement les articles non présents dans l'historique
unread_indices = [
    idx for idx in all_indices
    if idx not in read_indices
]

# Récupérer les scores des articles non consultés
unread_scores = similarity_scores[unread_indices]

# Trier les articles non consultés par score décroissant
sorted_positions = np.argsort(unread_scores)[::-1]

# Récupérer les indices réels des articles classés
top_indices = np.array(unread_indices)[sorted_positions]

print("===== FILTRAGE =====")

print("Articles dans le catalogue :", len(news_catalog))
print("Articles déjà consultés :", len(read_indices))
print("Articles non consultés :", len(unread_indices))

===== FILTRAGE =====
Articles dans le catalogue : 65238
Articles déjà consultés : 9
Articles non consultés : 65229


In [12]:
def recommend_content_based(
    history_indices,
    tfidf_matrix,
    news_catalog,
    k=10
):
    """
    Génère des recommandations Content-Based.

    Parameters
    ----------
    history_indices : list
        Indices des articles consultés.

    tfidf_matrix : csr_matrix
        Matrice TF-IDF des articles.

    news_catalog : DataFrame
        Catalogue des articles.

    k : int
        Nombre de recommandations souhaitées.

    Returns
    -------
    DataFrame
        Top-K articles recommandés avec leurs scores.
    """

    # Construire le profil utilisateur
    user_profile = build_user_profile(
        history_indices,
        tfidf_matrix
    )

    # Aucun historique exploitable : pas de profil
    if user_profile.nnz == 0:
        return pd.DataFrame(
            columns=[
                "news_id", "category", "subcategory",
                "title", "similarity_score"
            ]
        )

    # Calculer les similarités avec tous les articles
    scores = cosine_similarity(
        user_profile,
        tfidf_matrix
    ).flatten()

    # Exclure les articles déjà consultés
    read_indices = set(history_indices)

    candidate_indices = [
        idx for idx in range(len(news_catalog))
        if idx not in read_indices
    ]

    # Trier les candidats selon leurs scores
    sorted_indices = sorted(
        candidate_indices,
        key=lambda idx: scores[idx],
        reverse=True
    )

    # Sélectionner les K meilleurs articles
    top_indices = sorted_indices[:k]

    # Construire le résultat
    recommendations = news_catalog.iloc[
        top_indices
    ][
        ["news_id", "category", "subcategory", "title"]
    ].copy()

    recommendations["similarity_score"] = scores[top_indices]

    return recommendations.reset_index(drop=True)

In [13]:
recommendations = recommend_content_based(
    history_indices=example_history,
    tfidf_matrix=tfidf_matrix,
    news_catalog=news_catalog,
    k=10
)

print("===== RECOMMANDATIONS CONTENT-BASED =====")
print("Utilisateur :", example_user)

display(recommendations)

===== RECOMMANDATIONS CONTENT-BASED =====
Utilisateur : U13740


,news_id,category,subcategory,title,similarity_score
0,N61651,news,newspolitics,Former U.S. Sen. Kay Hagan dead at 66,0.272054
1,N61980,news,newspolitics,Former North Carolina US Sen Kay Hagan dies,0.257513
2,N59426,travel,travelnews,Four American Airlines flight attendants arres...,0.255579
3,N28476,news,newspolitics,"Former U.S. Sen. Kay Hagan, of North Carolina,...",0.252538
4,N54172,news,newspolitics,"Former U.S. Sen. Kay Hagan, of North Carolina,...",0.252538
5,N59336,news,newspolitics,Service to be held in NC for former US Sen. Ka...,0.249964
6,N61491,travel,travelnews,American Airlines Flight Attendants Arrested A...,0.242213
7,N55161,video,viral,Best Response Ever From a 'Wheel of Fortune' C...,0.233211
8,N28537,movies,movienews,"Robert Evans, Iconic 'Chinatown' Producer, Die...",0.213448
9,N628,news,newspolitics,South Carolina priest denied Joe Biden communi...,0.212142


In [14]:
recommended_ids = set(recommendations["news_id"])

# Articles déjà consultés
history_ids = set(
    news_catalog.iloc[list(set(example_history))]["news_id"]
)

print("===== VALIDATION DES RECOMMANDATIONS =====")

print("Nombre de recommandations :", len(recommendations))

print(
    "Articles déjà consultés recommandés :",
    len(recommended_ids.intersection(history_ids))
)

print(
    "Identifiants recommandés uniques :",
    recommendations["news_id"].nunique()
)

print(
    "Scores classés correctement :",
    recommendations["similarity_score"].is_monotonic_decreasing
)

assert len(recommendations) == min(
    10, len(news_catalog) - len(set(example_history))
)

assert len(recommended_ids.intersection(history_ids)) == 0

assert recommendations["similarity_score"].is_monotonic_decreasing

print("\nToutes les vérifications sont réussies !")

===== VALIDATION DES RECOMMANDATIONS =====
Nombre de recommandations : 10
Articles déjà consultés recommandés : 0
Identifiants recommandés uniques : 10
Scores classés correctement : True

Toutes les vérifications sont réussies !


In [16]:
# Charger les impressions du jeu dev
dev_impressions = pd.read_csv(
    "../data/processed/dev_impressions.csv.gz"
)

print("Impressions dev chargées avec succès !")

print("Dimensions :", dev_impressions.shape)

display(dev_impressions.head())

Impressions dev chargées avec succès !
Dimensions : (2740998, 6)


,impression_id,user_id,time,impressions,news_id,label
0,1,U80234,11/15/2019 12:37:50 PM,N28682-0,N28682,0
1,1,U80234,11/15/2019 12:37:50 PM,N48740-0,N48740,0
2,1,U80234,11/15/2019 12:37:50 PM,N31958-1,N31958,1
3,1,U80234,11/15/2019 12:37:50 PM,N34130-0,N34130,0
4,1,U80234,11/15/2019 12:37:50 PM,N6916-0,N6916,0


In [17]:
required_variables = [
    "dev_impressions",
    "dev_history",
    "tfidf_matrix",
    "news_catalog",
    "item_to_idx"
]

for variable in required_variables:
    print(
        variable,
        ":",
        "Disponible" if variable in globals() else "Manquante"
    )

dev_impressions : Disponible
dev_history : Disponible
tfidf_matrix : Disponible
news_catalog : Disponible
item_to_idx : Disponible


In [18]:
print("===== STRUCTURE DES IMPRESSIONS DEV =====")

print(dev_impressions.head())

print("\nNombre d'impressions :",
      dev_impressions["impression_id"].nunique())

print("\nNombre de candidats :",
      len(dev_impressions))

print("\nNombre de clics :",
      (dev_impressions["label"] == 1).sum())

print("\nNombre de non-clics :",
      (dev_impressions["label"] == 0).sum())

===== STRUCTURE DES IMPRESSIONS DEV =====
   impression_id user_id                    time impressions news_id  label
0              1  U80234  11/15/2019 12:37:50 PM    N28682-0  N28682      0
1              1  U80234  11/15/2019 12:37:50 PM    N48740-0  N48740      0
2              1  U80234  11/15/2019 12:37:50 PM    N31958-1  N31958      1
3              1  U80234  11/15/2019 12:37:50 PM    N34130-0  N34130      0
4              1  U80234  11/15/2019 12:37:50 PM     N6916-0   N6916      0

Nombre d'impressions : 73152

Nombre de candidats : 2740998

Nombre de clics : 111383

Nombre de non-clics : 2629615


In [19]:
def precision_at_k(relevance, k=10):
    """
    Precision@K = nombre de résultats pertinents dans Top-K / K
    """
    top_k = relevance[:k]
    return sum(top_k) / k


def recall_at_k(relevance, total_relevant, k=10):
    """
    Recall@K = nombre de résultats pertinents dans Top-K
                / nombre total de résultats pertinents
    """
    if total_relevant == 0:
        return 0.0

    top_k = relevance[:k]
    return sum(top_k) / total_relevant


def dcg_at_k(relevance, k=10):
    """
    Discounted Cumulative Gain.
    Les résultats pertinents en haut du classement
    ont plus de poids.
    """
    relevance = np.asarray(relevance[:k])

    if len(relevance) == 0:
        return 0.0

    discounts = np.log2(np.arange(2, len(relevance) + 2))

    return np.sum((2 ** relevance - 1) / discounts)


def ndcg_at_k(relevance, k=10):
    """
    Normalized Discounted Cumulative Gain.
    """
    actual_dcg = dcg_at_k(relevance, k)

    ideal_relevance = sorted(
        relevance,
        reverse=True
    )

    ideal_dcg = dcg_at_k(ideal_relevance, k)

    if ideal_dcg == 0:
        return 0.0

    return actual_dcg / ideal_dcg

In [20]:
history_lookup = (
    dev_history
    .set_index("impression_id")["history_indices"]
    .to_dict()
)

print("Nombre d'historiques disponibles :",
      len(history_lookup))

print("Exemple d'historique :",
      list(history_lookup.items())[0])

Nombre d'historiques disponibles : 73152
Exemple d'historique : (1, [6892, 30443, 14245, 29198, 28698, 16460, 1609, 27048, 31725, 27553, 11722, 23871, 27080, 1570, 32953])


In [21]:
def evaluate_impression(
    impression_df,
    history_lookup,
    tfidf_matrix,
    item_to_idx,
    k=10
):
    """
    Évalue le Content-Based pour une impression.

    Le classement est effectué uniquement sur les articles
    candidats de cette impression.
    """

    impression_id = impression_df["impression_id"].iloc[0]

    # Récupérer l'historique associé à cette impression
    history_indices = history_lookup.get(impression_id, [])

    # Construire le profil utilisateur
    profile = build_user_profile(
        history_indices,
        tfidf_matrix
    )

    if profile.nnz == 0:
        return None

    # Supprimer les éventuels candidats en double
    candidates = impression_df.drop_duplicates(
        subset="news_id"
    ).copy()

    # Convertir les IDs des articles en indices
    candidates["item_idx"] = (
        candidates["news_id"].astype(str).map(item_to_idx)
    )

    # Exclure les articles absents du catalogue
    candidates = candidates.dropna(
        subset=["item_idx"]
    ).copy()

    candidates["item_idx"] = candidates["item_idx"].astype(int)

    # Exclure les articles déjà consultés
    candidates = candidates[
        ~candidates["item_idx"].isin(set(history_indices))
    ]

    if len(candidates) == 0:
        return None

    # Calculer les similarités uniquement sur les candidats
    candidate_vectors = tfidf_matrix[
        candidates["item_idx"].tolist()
    ]

    scores = cosine_similarity(
        profile,
        candidate_vectors
    ).flatten()

    candidates["score"] = scores

    # Classer les articles par score décroissant
    candidates = candidates.sort_values(
        "score",
        ascending=False
    )

    # Récupérer les labels du classement
    relevance = candidates["label"].astype(int).tolist()

    total_relevant = int(candidates["label"].sum())

    # Calculer les métriques
    precision = precision_at_k(relevance, k)

    recall = recall_at_k(
        relevance,
        total_relevant,
        k
    )

    ndcg = ndcg_at_k(relevance, k)

    return {
        "impression_id": impression_id,
        "precision": precision,
        "recall": recall,
        "ndcg": ndcg,
        "candidate_count": len(candidates),
        "relevant_count": total_relevant
    }

In [22]:
sample_impressions = (
    dev_impressions["impression_id"]
    .drop_duplicates()
    .head(1000)
)

results = []

for impression_id in sample_impressions:

    impression_df = dev_impressions[
        dev_impressions["impression_id"] == impression_id
    ]

    result = evaluate_impression(
        impression_df=impression_df,
        history_lookup=history_lookup,
        tfidf_matrix=tfidf_matrix,
        item_to_idx=item_to_idx,
        k=10
    )

    if result is not None:
        results.append(result)


evaluation_df = pd.DataFrame(results)

print("===== PREMIÈRE ÉVALUATION CONTENT-BASED =====")

print("Impressions testées :", len(sample_impressions))
print("Impressions évaluées :", len(evaluation_df))

print("\nRésultats moyens :")

display(
    evaluation_df[
        ["precision", "recall", "ndcg"]
    ].mean().to_frame("Content-Based").T
)

===== PREMIÈRE ÉVALUATION CONTENT-BASED =====
Impressions testées : 1000
Impressions évaluées : 965

Résultats moyens :


,precision,recall,ndcg
Content-Based,0.083109,0.625044,0.383601


In [23]:
content_based_results = {
    "Model": "Content-Based",
    "Precision@10": evaluation_df["precision"].mean(),
    "Recall@10": evaluation_df["recall"].mean(),
    "NDCG@10": evaluation_df["ndcg"].mean(),
    "Evaluated impressions": len(evaluation_df)
}

content_based_results_df = pd.DataFrame([content_based_results])

display(content_based_results_df)

,Model,Precision@10,Recall@10,NDCG@10,Evaluated impressions
0,Content-Based,0.083109,0.625044,0.383601,965


In [24]:
import os

content_based_results = {
    "Model": "Content-Based",
    "Precision@10": 0.083109,
    "Recall@10": 0.625044,
    "NDCG@10": 0.383601,
    "Evaluated impressions": 965
}

content_based_results_df = pd.DataFrame([content_based_results])

# Créer le dossier s'il n'existe pas
os.makedirs("../data/processed", exist_ok=True)

# Enregistrer les résultats
content_based_results_df.to_csv(
    "../data/processed/evaluation_results.csv",
    index=False
)

print("Résultats enregistrés avec succès !")

Résultats enregistrés avec succès !


In [28]:
print("Fonction disponible :", "recommend_content_based" in globals())
print("Nombre total d'articles :", len(news_catalog))
print("Nombre d'impressions testées :", len(sample_impressions))

Fonction disponible : True
Nombre total d'articles : 65238
Nombre d'impressions testées : 1000


In [29]:
# ==========================================
# COVERAGE - CONTENT-BASED
# ==========================================

recommended_articles = set()

# Dictionnaire impression_id -> historique
history_lookup = dev_history.set_index(
    "impression_id"
)["history_indices"].to_dict()

# Parcourir les 1000 impressions
for impression_id in sample_impressions:

    history_indices = history_lookup.get(impression_id, [])

    # Générer les 10 recommandations
    recommendations = recommend_content_based(
        history_indices,
        tfidf_matrix,
        news_catalog,
        k=10
    )

    # Récupérer les identifiants des articles recommandés
    if recommendations is not None and not recommendations.empty:
        recommended_articles.update(
            recommendations["news_id"].astype(str).tolist()
        )

# Calcul du Coverage
total_articles = len(news_catalog)
coverage = len(recommended_articles) / total_articles

print("===== COVERAGE CONTENT-BASED =====")
print("Nombre d'articles distincts recommandés :", len(recommended_articles))
print("Nombre total d'articles :", total_articles)
print(f"Coverage : {coverage:.4%}")

===== COVERAGE CONTENT-BASED =====
Nombre d'articles distincts recommandés : 4265
Nombre total d'articles : 65238
Coverage : 6.5376%


In [30]:
# Ajouter le Coverage aux résultats Content-Based
content_based_results_df["Coverage"] = coverage

# Mettre à jour le fichier existant
content_based_results_df.to_csv(
    "../data/processed/evaluation_results.csv",
    index=False
)

display(content_based_results_df)

print("Résultats mis à jour avec succès !")

,Model,Precision@10,Recall@10,NDCG@10,Evaluated impressions,Coverage
0,Content-Based,0.083109,0.625044,0.383601,965,0.065376


Résultats mis à jour avec succès !


In [32]:
# Charger les interactions positives du train
train_unique_interactions = pd.read_csv(
    "../data/processed/train_interactions.csv"
)

print("Interactions train chargées :", len(train_unique_interactions))
print("Colonnes :", train_unique_interactions.columns.tolist())

display(train_unique_interactions.head())

Interactions train chargées : 234468
Colonnes : ['user_id', 'news_id']


,user_id,news_id
0,U13740,N55689
1,U91836,N17059
2,U73700,N23814
3,U34670,N49685
4,U8125,N8400


In [33]:
clicked_articles = set(
    train_unique_interactions["news_id"].astype(str)
)

cold_start_articles = (
    set(news_catalog["news_id"].astype(str))
    - clicked_articles
)

print("===== ANALYSE DES ARTICLES COLD START =====")
print("Articles dans le catalogue :", len(news_catalog))
print("Articles avec au moins un clic :", len(clicked_articles))
print("Articles sans aucun clic :", len(cold_start_articles))

print(
    f"Proportion Cold Start : "
    f"{len(cold_start_articles) / len(news_catalog) * 100:.2f}%"
)

===== ANALYSE DES ARTICLES COLD START =====
Articles dans le catalogue : 65238
Articles avec au moins un clic : 7713
Articles sans aucun clic : 57525
Proportion Cold Start : 88.18%


In [34]:
# ==========================================
# COLD START RECOMMENDATIONS
# ==========================================

recommended_cold_items = (
    recommended_articles & cold_start_articles
)

total_recommended = len(recommended_articles)
total_cold_items = len(cold_start_articles)
total_recommended_cold = len(recommended_cold_items)

# Part des recommandations qui sont Cold Start
cold_recommendation_percentage = (
    total_recommended_cold / total_recommended * 100
)

# Part du catalogue Cold Start recommandée
cold_catalog_coverage = (
    total_recommended_cold / total_cold_items * 100
)

print("===== RÉSULTATS COLD START CONTENT-BASED =====")

print("Articles distincts recommandés :", total_recommended)
print("Articles Cold Start disponibles :", total_cold_items)
print("Articles Cold Start recommandés :", total_recommended_cold)

print(
    f"Part des recommandations Cold Start : "
    f"{cold_recommendation_percentage:.2f}%"
)

print(
    f"Coverage des articles Cold Start : "
    f"{cold_catalog_coverage:.2f}%"
)

===== RÉSULTATS COLD START CONTENT-BASED =====
Articles distincts recommandés : 4265
Articles Cold Start disponibles : 57525
Articles Cold Start recommandés : 3725
Part des recommandations Cold Start : 87.34%
Coverage des articles Cold Start : 6.48%


In [35]:
import os
import pandas as pd

os.makedirs("../data/processed", exist_ok=True)

# Résultats principaux Content-Based
content_based_results = {
    "Model": "Content-Based",
    "Precision@10": 0.083109,
    "Recall@10": 0.625044,
    "NDCG@10": 0.383601,
    "Coverage": coverage,
    "Evaluated impressions": len(evaluation_df),
    "Sample impressions": len(sample_impressions)
}

content_based_results_df = pd.DataFrame([content_based_results])

content_based_results_df.to_csv(
    "../data/processed/evaluation_results.csv",
    index=False
)

# Résultats Cold Start
cold_start_results = {
    "Model": "Content-Based",
    "Total articles": len(news_catalog),
    "Articles with interactions": len(clicked_articles),
    "Cold Start articles": len(cold_start_articles),
    "Cold Start proportion (%)": len(cold_start_articles) / len(news_catalog) * 100,
    "Distinct recommended articles": total_recommended,
    "Recommended Cold Start articles": total_recommended_cold,
    "Cold Start recommendation proportion (%)": cold_recommendation_percentage,
    "Cold Start catalog coverage (%)": cold_catalog_coverage
}

cold_start_results_df = pd.DataFrame([cold_start_results])

cold_start_results_df.to_csv(
    "../data/processed/cold_start_results.csv",
    index=False
)

print("Résultats Content-Based sauvegardés !")
print("\nRésultats principaux :")
display(content_based_results_df)

print("\nRésultats Cold Start :")
display(cold_start_results_df)

Résultats Content-Based sauvegardés !

Résultats principaux :


,Model,Precision@10,Recall@10,NDCG@10,Coverage,Evaluated impressions,Sample impressions
0,Content-Based,0.083109,0.625044,0.383601,0.065376,965,1000



Résultats Cold Start :


,Model,Total articles,Articles with interactions,Cold Start articles,Cold Start proportion (%),Distinct recommended articles,Recommended Cold Start articles,Cold Start recommendation proportion (%),Cold Start catalog coverage (%)
0,Content-Based,65238,7713,57525,88.177136,4265,3725,87.338804,6.475445


In [36]:
# Sauvegarder l'échantillon commun
sample_impressions_df = pd.DataFrame({
    "impression_id": sample_impressions
})

sample_impressions_df.to_csv(
    "../data/processed/evaluation_sample_1000.csv",
    index=False
)

# Sauvegarder les impressions effectivement évaluées
evaluation_df[["impression_id"]].to_csv(
    "../data/processed/evaluated_impressions_965.csv",
    index=False
)

print("Échantillon et impressions évaluées sauvegardés !")

Échantillon et impressions évaluées sauvegardés !
